1. Establish a connection between Python and the Sakila database.

In [ ]:
#!pip install mysql-connector-python

In [5]:
#connection to a Sakila database
import mysql.connector
from mysql.connector import Error
from getpass import getpass
from sqlalchemy import create_engine

def create_mysql_connection(host, user, database):
    try:
        password = getpass("Introduce tu contraseña MySQL: ")  # poner la contraseña de forma segura
        conn = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database
        )
        if conn.is_connected():
            print(f"Conectado con éxito a {database}")
        #Crear engine SQLAlchemy para Pandas usando el mismo password
            engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")
            print("Engine para Pandas creado con éxito")
            return conn, engine  # devolvemos ambas cosas
        
    except Error as e:
        print(f"Error de conexión: {e}")
        return None, None

In [6]:
#Usa tu usuario, host y la base sakila

connection, engine = create_mysql_connection(
    host="localhost",
    user="root",        # puedes cambiar a otro usuario si quieres
    database="sakila")

Conectado con éxito a sakila
Engine para Pandas creado con éxito


2. Write a Python function called rentals_month that retrieves rental data for a given month and year (passed as parameters) from the Sakila database as a Pandas DataFrame.

The function should take in three parameters:
 - engine: an object representing the database connection engine to be used to establish a connection to the Sakila database.
 - month: an integer representing the month for which rental data is to be retrieved.
 - year: an integer representing the year for which rental data is to be retrieved.

The function should execute a SQL query to retrieve the rental data for the specified month and year from the rental table in the Sakila database, and return it as a pandas DataFrame.


In [9]:
import pandas as pd

def rentals_month(engine, month, year):
    """
    Returns a DataFrame with rental data filtered by month and year.
    """

    query = """
        SELECT *
        FROM rental
        WHERE MONTH(rental_date) = %s
        AND YEAR(rental_date) = %s;
    """
    
    # 👇 Usamos parámetros seguros en vez de f-string
    df_rentals = pd.read_sql(query, engine, params=(month, year))
    return df_rentals


In [ ]:
#EJEMPLO USO
df_rentals = rentals_month(engine, 6, 2005)
df_rentals.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1158,2005-06-14 22:53:33,1632,416,2005-06-18 21:37:33,2,2006-02-15 21:30:53
1,1159,2005-06-14 22:55:13,4395,516,2005-06-17 02:11:13,1,2006-02-15 21:30:53
2,1160,2005-06-14 23:00:34,2795,239,2005-06-18 01:58:34,2,2006-02-15 21:30:53
3,1161,2005-06-14 23:07:08,1690,285,2005-06-21 17:12:08,1,2006-02-15 21:30:53
4,1162,2005-06-14 23:09:38,987,310,2005-06-23 22:00:38,1,2006-02-15 21:30:53


3. Develop a Python function called rental_count_month that takes the DataFrame provided by rentals_month as input along with the month and year and returns a new DataFrame containing the number of rentals made by each customer_id during the selected month and year.

The function should also include the month and year as parameters and use them to name the new column according to the month and year, for example, if the input month is 05 and the year is 2005, the column name should be "rentals_05_2005".

*Hint: Consider making use of pandas [groupby()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)*

In [12]:
#function called rental_count_month

import pandas as pd

def rental_count_month(df_rentals, month, year):
    """
    Returns a DataFrame with the number of rentals per customer_id
    for a given month and year.
    
    The output column is named rentals_MM_YYYY.
    
    Parameters:
        df_rentals (DataFrame): DataFrame from rentals_month().
        month (int): Month number (1-12).
        year (int): Year (e.g., 2005).
        
    Returns:
        DataFrame: customer_id + number of rentals in that period.
    """
    
    # Format month with 2 digits (e.g., 05 instead of 5)
    month_str = f"{month:02d}"
    col_name = f"rentals_{month_str}_{year}"
    
    # Group by customer_id and count rentals
    result = (
        df_rentals.groupby("customer_id")["rental_id"]
        .count()
        .reset_index(name=col_name)
    )
    
    return result


In [13]:
#EJEMPLO USO

df_counts = rental_count_month(df_rentals, 6, 2005)
df_counts.head()



,customer_id,rentals_06_2005
0,1,7
1,2,1
2,3,4
3,4,6
4,5,5



4. Create a Python function called compare_rentals that takes two DataFrames as input containing the number of rentals made by each customer in different months and years.

The function should return a combined DataFrame with a new 'difference' column, which is the difference between the number of rentals in the two months.

In [18]:
#function called compare_rentals

import pandas as pd

def compare_rentals(df_month1, df_month2):
    """
    Compares two rental count DataFrames (customer_id + rentals_MM_YYYY)
    and returns a merged DataFrame with a new 'difference' column.
    
    The difference column shows how many more (or fewer) rentals a customer 
    made in the second month compared to the first.
    """
    
    # Merge ambos DataFrames por customer_id
    df_compare = pd.merge(df_month1, df_month2, on="customer_id", how="outer").fillna(0)
    
    # Nombre dinámico de columnas (segundo - primero)
    col1 = df_month1.columns[1]
    col2 = df_month2.columns[1]
    
    # Crear columna difference
    df_compare["difference"] = df_compare[col2] - df_compare[col1]

    # Convert all rental columns to int
    for col in df_compare.columns:
        if col != "customer_id":
            df_compare[col] = df_compare[col].astype(int)
    
    return df_compare


In [19]:
#EJEMPLO USO

df_06 = rental_count_month(rentals_month(engine, 6, 2005), 6, 2005)
df_07 = rental_count_month(rentals_month(engine, 7, 2005), 7, 2005)

df_compare = compare_rentals(df_06, df_07)
df_compare.head()


,customer_id,rentals_06_2005,rentals_07_2005,difference
0,1,7,12,5
1,2,1,14,13
2,3,4,13,9
3,4,6,5,-1
4,5,5,16,11
